# RGCA Real Generation Milestone

Run this after the full Kaggle retrieval pipeline has produced:

- `/kaggle/working/rgca_hydrated_subset/mimic_subset.jsonl`
- `/kaggle/working/rgca_experiments/biomedclip_retrieval_validation_v0/retrieval_results.jsonl`
- `/kaggle/working/rgca_experiments/biomedclip_retrieval_validation_v0/mismatch_results.jsonl`

For a real VLM run, set Kaggle secret `RGCA_VLM_MODEL_ID` to a HuggingFace vision-language checkpoint. If you leave `GENERATOR_BACKEND = 'retrieval_copy_stress'`, this notebook runs a cheap protocol/debug generation pass only.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

PROJECT_ROOT = Path('/kaggle/working/RGCA')
SUBSET_PATH = Path('/kaggle/working/rgca_hydrated_subset/mimic_subset.jsonl')
RETRIEVAL_RESULTS = Path('/kaggle/working/rgca_experiments/biomedclip_retrieval_validation_v0/retrieval_results.jsonl')
MISMATCH_RESULTS = Path('/kaggle/working/rgca_experiments/biomedclip_retrieval_validation_v0/mismatch_results.jsonl')
OUTPUT_DIR = Path('/kaggle/working/rgca_experiments/real_generation_v0')
EVIDENCE_ZIP = Path('/kaggle/working/rgca_real_generation_evidence_v0.zip')

# Use 'hf_vlm' for real inference. Use 'retrieval_copy_stress' only for a cheap protocol check.
GENERATOR_BACKEND = os.environ.get('RGCA_GENERATOR_BACKEND', 'retrieval_copy_stress')
MODEL_ID = os.environ.get('RGCA_VLM_MODEL_ID', '')
LIMIT = int(os.environ.get('RGCA_GENERATION_LIMIT', '20'))

def run(command, cwd=PROJECT_ROOT, env=None):
    print('+', ' '.join(str(part) for part in command))
    return subprocess.run(command, cwd=str(cwd), env=env, check=True)

if not PROJECT_ROOT.exists():
    run(['git', 'clone', 'https://github.com/pidoxy/RGCA.git', str(PROJECT_ROOT)], cwd=Path('/kaggle/working'))

run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])

required = {
    'subset': SUBSET_PATH,
    'retrieval_results': RETRIEVAL_RESULTS,
    'mismatch_results': MISMATCH_RESULTS,
}
missing = {name: str(path) for name, path in required.items() if not path.exists()}
if missing:
    raise FileNotFoundError(
        'Run notebooks/kaggle_full_research_pipeline.ipynb first, or attach/copy these artifacts. Missing: '
        + json.dumps(missing, indent=2)
    )

if GENERATOR_BACKEND == 'hf_vlm':
    if not MODEL_ID:
        raise ValueError('Set Kaggle secret/environment variable RGCA_VLM_MODEL_ID before real VLM generation.')
    run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'accelerate', 'pillow'])

command = [
    sys.executable,
    'scripts/run_generation_from_retrieval.py',
    '--subset', str(SUBSET_PATH),
    '--retrieval-results', str(RETRIEVAL_RESULTS),
    '--mismatch-results', str(MISMATCH_RESULTS),
    '--output-dir', str(OUTPUT_DIR),
    '--generator', GENERATOR_BACKEND,
    '--mode', 'all',
    '--limit', str(LIMIT),
]
if GENERATOR_BACKEND == 'hf_vlm':
    command.extend(['--model-id', MODEL_ID, '--require-real-generator', '--require-images'])

run(command)

if EVIDENCE_ZIP.exists():
    EVIDENCE_ZIP.unlink()
shutil.make_archive(str(EVIDENCE_ZIP.with_suffix('')), 'zip', OUTPUT_DIR)

manifest = OUTPUT_DIR / 'generation_manifest.json'
print('\nGeneration milestone complete.')
print('manifest:', manifest, 'exists=', manifest.exists())
print('zip:', EVIDENCE_ZIP, 'exists=', EVIDENCE_ZIP.exists(), 'size_mb=', round(EVIDENCE_ZIP.stat().st_size / 1_000_000, 3))
print(json.dumps(json.loads(manifest.read_text()), indent=2)[:5000])


## Inspect Generated Evidence

Use this cell to confirm the generated files exist and preview evaluation summaries.

In [ ]:
from pathlib import Path
import json

root = Path('/kaggle/working/rgca_experiments/real_generation_v0')
for path in sorted(root.rglob('*')):
    if path.is_file():
        print(path, 'size_mb=', round(path.stat().st_size / 1_000_000, 3))

for summary in sorted(root.glob('evaluation_*/evaluation_summary.json')):
    print('\n', summary)
    print(json.dumps(json.loads(summary.read_text()), indent=2))
